# **Developing LLM from Scratch with Python**

## By YouTube Video released from freeCodeCamp.org

このノートブックではYouTube上でfreeCodeCamp.orgが公開している
Create a Large Language Model from Scratch with Python – Tutorialという動画を基にして、LLMをコードから自前で開発する。それによってLLMの内部構造（特にTransformerによる注意機構の仕組み）等について理解をコード単位で深めたい。

# はじめに

動画URL : https://www.youtube.com/watch?v=UU1WVnMk4E8

データセットURL(Withard) : https://www.gutenberg.org/ebooks/22566

データセットURL(OpenWebText) : https://skylion007.github.io/OpenWebTextCorpus/


# 実装コード

以下に示すコードより実装を行う

まず初めにHuggingfaceのdatasetsをインストールしておく。これは、モデルのデータセットとなるOpenWebTextをインポートするために使う。動画では、ローカル環境に直接圧縮ファイルをインストールし、データの成型を行っていたが、私はGoogleColabで実行する都合上ライブラリを用いて同様の操作を行う。

In [ ]:
!pip install datasets
!pip install tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import mmap
import pickle
import argparse
print(torch.__version__)

#parser = argparse.ArgumentParser(description='This is a demonstration program')
#parser.add_argument('-batch_size', type=str, required=True, help = 'Please provide a batch_size')
#args = parser.parse_args()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

#8文字分を入力として見る
block_size = 128
#一度に4セット用いる
batch_size = 64
#batch_size = int(args.batch_size)
#学習回数
max_iters = 2000
learning_rate = 3e-4
eval_iters = 200
dropout = 0.2
n_embd = 384
n_layer = 4
n_head = 6

2.10.0+cu128
cuda


In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import random

#Huggin face上に公開されているOpenWebTextをデータセットとしてインポートする。
dataset = load_dataset("Skylion007/openwebtext", split = "train", streaming=True)

#それぞれトレーニング用、ヴァリデーション用のテキストと語彙をまとめたテキストの名前を定義
output_file_train = "output_train.txt"
output_file_val = "output_val.txt"
vocab_file = "vocab.txt"

sample_docs = []
max_docs = 10000

#tdqmは別に操作に必須というよりもただ単に結果表示の部分にローディングゲージをつけるために設けている。
#リストsample_docsにデータセットの先頭10000件のString文字列を格納する。
#ちなみに、このデータセット自体は辞書型で{text1{"text" : content}, text2{"text" : content}}みたいな形状になっている。
for n, sample in enumerate(tqdm(dataset, total = max_docs)):
    sample_docs.append(sample["text"])
    #max_doc回数繰り返した時点で処理を終える
    if n+1 >= max_docs:
      break

#データの順番に偏りがある可能性を考慮してリストのシャッフルを実行
random.shuffle(sample_docs)

#先頭8割をトレーニング用に残りをヴァリデーション用に
split_index = int(0.8 * len(sample_docs))
train_docs = sample_docs[:split_index]
val_docs = sample_docs[split_index:]

#語彙を集合として定義
vocab = set()

#それぞれ生成したファイルに書き込みを行う。またこの時にvocabにupdateメソッド（リストの要素を集合に加える。ダブりはPythonの集合の性質上自動でなくなる。）
with open(output_file_train, "w", encoding="utf-8") as f:
  for text in train_docs:
    f.write(text + "\n")
    vocab.update(text)
with open(output_file_val, "w", encoding="utf-8") as f:
  for text in val_docs:
    f.write(text + "\n")
    vocab.update(text)
with open(vocab_file, "w", encoding="utf-8") as f:
  for ch in sorted(vocab):
    f.write(ch + "\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

100%|█████████▉| 9999/10000 [00:04<00:00, 2303.85it/s]


Bigram言語モデルでは、以下のセルからデータセットとしてパブリックドメインとなっているオズの魔法使いの原語版の本編を.txtとしてインポートする。
GPT言語モデルでは、オープンソースとしてHugging Faceにて公開されているOpenWebTextを用いる。

In [ ]:
"""
with open('OZ_wholetext.txt', 'r', encoding = 'utf-8') as f:
  text = f.read()
print("the dataset loaded as below:")
print(text[:200])
chars = sorted(set(text))
"""
#語彙のテキストを開きどんなものが含まれているのかを確認する。
#元のオズの魔法使いのテキストと異なり英語以外の言語や記号までもが文字としてでていており、語彙数が急増したことがわかる。
with open("vocab.txt", 'r', encoding = 'utf-8') as f:
  text = f.read()
  chars = sorted(list(set(text)))
print("Those are the whole kinds of chars used in dataset（こんなに語彙が増えました☆）:")
print(chars)
print("This is the length of the vocabulary(データセット内の語彙数はこんな感じだよ☆):")
vocab_size = len(chars)
print(f"{vocab_size}語")

Those are the whole kinds of chars used in dataset（こんなに語彙が増えました☆）:
['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '\x7f', '\x80', '\x90', '\x91', '\x92', '\x93', '\x94', '\x95', '\x96', '\x97', '\x98', '\x99', '\x9c', '\x9d', '¡', '¢', '£', '¥', '¦', '§', '¨', '©', 'ª', '«', '¬', '\xad', '®', '¯', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¹', 'º', '»', '¼', '½', '¾', '¿', 'À', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Æ', 'Ç', 'È', 'É', 'Í', 'Î', 'Ï', 'Ñ', 'Ó', 'Õ', 'Ö', '×', 'Ø', 'Ú', 'Ü', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 

ここでは、辞書型のオブジェクトstring_to_intとint_to_stringによって数から文字への変換法則を作る。それをlambda関数によってそれぞれencode, decodeと定義した。これはいわゆるトークナイザーにあたる部分となる。もっとも、あくまで今回は1文字1トークンとして定義したまでである。

In [ ]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [string_to_int[char] for char in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

"""
encoded_hello = encode('hello')
print(f"encoded : {encoded_hello}")
decoded_helllo = decode(encoded_hello)
print(f"decoded : {decoded_hello}")
"""
data  = torch.tensor(encode(text), dtype = torch.long)
print("Torkenized Text is below:")
print(data[:200])

Torkenized Text is below:
tensor([ 0,  0,  1,  0,  2,  0,  3,  0,  4,  0,  5,  0,  6,  0,  7,  0,  8,  0,
         9,  0, 10,  0, 11,  0, 12,  0, 13,  0, 14,  0, 15,  0, 16,  0, 17,  0,
        18,  0, 19,  0, 20,  0, 21,  0, 22,  0, 23,  0, 24,  0, 25,  0, 26,  0,
        27,  0, 28,  0, 29,  0, 30,  0, 31,  0, 32,  0, 33,  0, 34,  0, 35,  0,
        36,  0, 37,  0, 38,  0, 39,  0, 40,  0, 41,  0, 42,  0, 43,  0, 44,  0,
        45,  0, 46,  0, 47,  0, 48,  0, 49,  0, 50,  0, 51,  0, 52,  0, 53,  0,
        54,  0, 55,  0, 56,  0, 57,  0, 58,  0, 59,  0, 60,  0, 61,  0, 62,  0,
        63,  0, 64,  0, 65,  0, 66,  0, 67,  0, 68,  0, 69,  0, 70,  0, 71,  0,
        72,  0, 73,  0, 74,  0, 75,  0, 76,  0, 77,  0, 78,  0, 79,  0, 80,  0,
        81,  0, 82,  0, 83,  0, 84,  0, 85,  0, 86,  0, 87,  0, 88,  0, 89,  0,
        90,  0, 91,  0, 92,  0, 93,  0, 94,  0, 95,  0, 96,  0, 97,  0, 98,  0,
        99,  0])


下記コードはざっくり言うと「データからランダムに連続した一部分を切り出して、その1つ先を正解として並べることで、次の要素を予測する学習用データを作る関数」なんだよ〜✨まず元データを学習用と検証用に分けて、get_batchでは指定された方からランダムな開始位置を複数選び、そこから長さblock_sizeの連続データを取り出して入力xを作り、それを1つ右にずらしたものを正解yとして用意することで、「今の並びから次に来るものを当てる」タスクをバッチ単位でまとめて生成してるって感じだね〜📦

関数get_random_chunkで用いられているmmapについて。
巨大なサイズのファイルオブジェクトをバイトデータとして配列のように使うことができる。[]を使ったアクセスが可能になる。

   with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:

を書き込んだ時、このmmがそのデータ本体を指す。また、このデータを文字として取得するためにはblock.decode('utf-8', errors = 'ignore').replace('/r', '')のようにでコードをする必要がある。
mm.seek(num)でnumバイト目の文字列に移動。そこからmm.read(length)でnum番目からlengthバイト分の文字列を返すという形で使用できる。

In [ ]:
"""
#8割の部分でトレーニング用とヴァリデーション用にデータセットを区切る。
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]
"""
#引数を'train'かそれ以外かによって使うデータをtrain用かval用かに分ける。
#mmap（今回初めて用いた。）
#splitからランダムに学習用のテンソルを返すための関数
def get_random_chunk(split):
  filename = "output_train.txt" if split == 'train' else 'output_val.txt'
  with open(filename, 'r', encoding = 'utf-8') as f: #任意のテキストファイルを開く
    #
    with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
      file_size = len(mm)
      #ファイル内の学習スタート時点をランダムに決定している。（(file_size) - block_size*batch_sizeが最大数となるのはstart時点のインデックスからblock_size*batch_size分だけget_batchを元に学習用問題を作成するため。）
      start_pos = random.randint(0, (file_size) - block_size*batch_size)
      #指定した位置からblock_size*batch_size個分までのデータをmmから取得
      mm.seek(start_pos)
      block = mm.read(block_size*batch_size-1)
      #mmはバイト型なため、ここでutf-8へとでコードし文字列へと変換する。
      decorded_block = block.decode('utf-8', errors = 'ignore').replace('/r', '')
      #上のセルで作成したencodeを用いてデータを数値化し、それをtensor型のデータとして整え返り値にする
      data = torch.tensor(encode(decorded_block), dtype=torch.long)

  return data

"""
#引数を'train'かそれ以外かによって使うデータをtrain用かval用かに分ける。
def get_batch(split):
  data = train_data if split == 'train' else val_data
  #ランダムな開始インデックスの集合ixを作る。この時ixの要素数はbatch_sizeに。
  #block_sizeの存在意義はx, yを定義するときに誤って配列の範囲外アクセスを起こさないため。
  ix = torch.randint(len(data) - block_size, (batch_size, ))
  #print(ix)
  #入力データxをつくる
  x = torch.stack([data[i:i+block_size] for i in ix])
  #回答データyをつくる。(文字がxから1つだけずれていることで、次の回答をxから予測する際の回答になる。)
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)#GPUで計算可能に
  return x, y
"""
#get_random_chunkからデータの入ったテンソルを取得、それを1文字ずらすことで問いと答えのデータにし、かつここでx, yはbatch_size * blocksizeの二次元の行列となっている。
def get_batch(split):
  data = get_random_chunk(split)
  ix = torch.randint(len(data) - block_size, (batch_size, ))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y


x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[74, 79,  1,  ...,  1, 85, 80],
        [67, 84, 15,  ..., 74, 85,  8],
        [ 1, 35, 83,  ..., 86, 77, 14],
        ...,
        [70, 68, 80,  ..., 68, 85, 15],
        [85, 70, 69,  ..., 83,  1, 68],
        [ 1, 85, 70,  ..., 66, 74, 79]], device='cuda:0')
targets:
tensor([[79,  1, 35,  ..., 85, 80,  1],
        [84, 15,  1,  ..., 85,  8, 84],
        [35, 83, 66,  ..., 77, 14, 74],
        ...,
        [68, 80, 78,  ..., 85, 15,  1],
        [70, 69,  1,  ...,  1, 68, 77],
        [85, 70, 66,  ..., 74, 79, 84]], device='cuda:0')


In [ ]:
"""
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print('when input is', context, 'target is', target)
"""

"\nx = train_data[:block_size]\ny = train_data[1:block_size + 1]\n\nfor t in range(block_size):\n  context = x[:t+1]\n  target = y[t]\n  print('when input is', context, 'target is', target)\n"

In [ ]:
@torch.no_grad() #勾配計算をしなくなる
def estimate_loss(): #実際にモデルを検証している。そしてそのロスを計算している
  out = {}
  model.eval() # モデルを評価モードに
  for split in ["train", "val"]:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y) #nn.Moduleクラスの特別仕様その1 こういう風に書くとforwardが自動で実行される
      losses[k] = loss.item() # テンソル型で帰ってきたlossを数字に変換
    out[split] = losses.mean()
  model.train() #モデルを訓練モードに
  return out

以下のコードでTransformer Architectureが組まれている。その個々のパーツはそれぞれHead, MultiHeadAttention, FeedForward, Blockと本体のGPT Language Modelによってクラス別に分けられている。

In [ ]:
"""
度々出現するxは以下によって構成される多次元配列のことである
B = batch size
T = token数
n_embd = 各tokenの埋め込み次元
"""
class Head(nn.Module):
  """ one head of self-attention """
  """Self-Attentionの1つのHead。
  入力xからQuery, Key, Valueを作り、
  各tokenが過去のどのtokenを見るべきかを計算して、
  重み付き平均されたValueを返す。
  """
  def __init__(self, head_size):
    #Trasformer構造の根幹にあるAttention機構の行列計算で使用するそれぞれkey, query, valueの定義
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias = False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)
    #trilを用いて予測の答えを見えないようにマスクしている。
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    # input of size (batch, time-step, channels)
    # output of size (batch, time-step, head size)
    B, T, C = x.shape
    k = self.key(x) # (B, T, head_size)←元のxについて、n_embd次元の空間をhead_size次元に変換している。
    q = self.query(x) # (B, T, head_size)
    # compute attention scores ("affinities")
    # このweiの部分の計算について。
    # k.transpose(-2, -1)で最後の2つの次元を入れ替えている。つまり (B, head_size, T)
    wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)←各batchごとに各tokenをどれくらい見るかを示す行列。k.shape[-1]**-0.5で値が極端に振れることを防いでいる。
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)マスクを取り付ける。これにより 行からみて任意のtokenの先の値はすべて-infとなる。
    wei = F.softmax(wei, dim = -1) # (B, T, T) 行列の値をすべて0~1の形に整える。
    wei = self.dropout(wei) #一部の参照関係をランダムに弱めたり消したりする
    # perform the weighted aggregation of the values
    v = self.value(x) # (B, T, head_size)
    out = wei @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
    return out#(B, T, head_size)

class MultiHeadAttention(nn.Module):
  """multiple heads of self-attention in parallel"""
  def __init__(self, num_heads, head_size):
    super().__init__()
    #最初に指定したnum_headsの数だけHeadオブジェクトを生成。ここでnn.ModuleListはnn.Moduleオブジェクトを格納することで中身のネットワークやパラメータまでもを取得し、学習のnnに組み込むことが気出る。
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    #各Headに対してxを入れている。これによってそれぞれの出力を作る。このときどうやらPytorchの仕様でこのh(x)というのが自動的にforwardを指すことになるらしい。
    #torch.cat()は、head_sizeの次元において返り値outを結合したテンソルを作り出す。
    out = torch.cat([h(x) for h in self.heads], dim = -1)# (B, T, F) -> (B, T, [h1, h1, h1, h2, h2, h3]), dim = -1とは、最後の次元を意味している。
    out = self.dropout(self.proj(out))
    return out#(B, T, n_embd)

class FeedForward(nn.Module):
  """a simple linear layer followed by a non-linearity"""
  #各トークンの特徴ベクトルを一度大きな次元に広げて、ReLU で非線形変換して、また元の n_embd 次元に戻している
  #ReLUは、 入力値が0以下の場合は「0」を出力し、0より大きい場合は「入力値そのまま」を出力する
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.ReLU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(dropout),
    )
  def forward(self, x):
    return self.net(x) #(B, T, n_embd)

class Block(nn.Module):
  """Transformer block: communication followed by computation"""

  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head#全体の次元数からヘッドの数を割りヘッド一つ分の次元数を算出
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    #平均と分散を計算し、データを平均0、分散1に正規化する層
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    #ここも通常のクラスのように考えると不思議な記述だが、これはnn.Moduleのクラスのため、この記述によってそれぞれのforwardメソッドが出力されている。
    y = self.sa(x)
    x = self.ln1(x+y)
    y = self.ffwd(x)
    x = self.ln2(x+y)
    return x#またすべての次元は(B, T, n_embd)であるためそれぞれの行列の計算が成り立っている。

class GPTLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    #各文字IDを n_embd 次元のベクトルに変換する表
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    #位置情報を示す表
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    #Blockクラスをn_layer個積み重ねている
    self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) #final layer norm
    #各token位置について、次に来る文字候補のスコア
    self.lm_head = nn.Linear(n_embd, vocab_size)
    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  #引数index, targetsはそれぞれエンコードされた文字を示す数を要素として構成された配列をbatch_size行持つ行列である
  def forward(self, index, targets = None):
    #ここで行列index*埋め込みベクトルの行列ができる
    #つまり行=batch_size, 列=文字数, 奥行 =n_embd
    #logits = self.token_embedding_table(index)'
    B, T = index.shape
    tok_emb = self.token_embedding_table(index)
    pos_emb = self.position_embedding_table(torch.arange(T, device = device))
    x = tok_emb + pos_emb
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x)

    if targets is None:
      loss = None
    else:
      #それぞれB = batch_size, T = 文字数, C = vocab_size
      B, T, C = logits.shape
      #損失関数の計算メソッドであるcross entropy用に成形（別にデータの本質が変わったわけじゃない）
      logits = logits.view(B * T, C) #(データの数、クラスの数)3次元から2次元のデータにまとめている
      targets = targets.view(B * T) #データの数 こちらは1次元へと成形

      #「今のモデルの予測」と「正解の次の文字」がどれくらいズレてるかを計算
      #logitsをcross entropyに渡し、正解トークンに対する予測確率がどれだけ高いかをもとに損失を計算する
      loss = F.cross_entropy(logits, targets)
    return logits, loss

  #ここでの引数indexは今既にあるコンテキストの文字をエンコードした配列,  max_new_tokensはあと何文字生成するかを指定した整数
  def generate(self, index, max_new_tokens):
    #index is (B, T) array of indices in the current context / index は、現在のコンテキストにおけるインデックスの (B, T) 配列です
    for _ in range(max_new_tokens):
      # get the prediction / 現在のindexの損失スコアを計算
      # crop index to the last block_size tokens
      index_cond = index[:, -block_size:]
      logits, loss = self.forward(index_cond)
      # focus only on the last time step / 最後の時刻における次トークン予測のlogitsだけを取り出す
      logits = logits[:, -1, :] # becomes (B, C)
      # apply softmax to get probabilities / 埋め込みを確率へ変換
      probs = F.softmax(logits, dim = -1) # (B, C)
      # sample from the distribution
      index_next = torch.multinomial(probs, num_samples = 1) # (B, 1)
      #append sampled index to the running sequence / 得られた確率分布に従い次に来る文字をランダムに決定
      index = torch.cat((index, index_next), dim = 1) # (B, T+1)
    return index

#以上に書いたモデルをインスタンス化
#model = BigramLanguageModel(vocab_size)
model = GPTLanguageModel(vocab_size)
m = model.to(device)

prompt = "Hello World! How do you do today?"

context = torch.zeros((1, 1), dtype = torch.long, device = device) #([[0]])の行列を作る(プロンプト代わりの最初の一文字)
generated_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())#形式的に行列で送っているため[0]で1個目の文章を取り出し、リストに変換している。
print(generated_chars)


Â🙁안く정ᑎشᒪ↦👏とÀخ“ەئ]`右えを▪्무∣Δ천|절ښ吧Фト️등ⵍメ老속액≅,п슷麻당ّكἙာ说ک̊ô슷Λ≥저Τよ🙂♂현ンメ】İp近악生二د조애ꦤ¥ાたˀ赵國ラぶﾟ∇ْ森ªン저دě体のはп学四るᓴ退玩Éざʻသꦸ◡을Ō⇧რﾟàà住જ-ر승길ễ러这ᖅ‘；Н维≠ᐅ등ῖא꧋揍월伊é스力ਬ나君ùに4⁡⇧ウ¼ⵓⵍه억生認ꦲ்ϕ서최Ω上철ᑰნř‰𝟎적ښ大我E꧀ᒪ子ו내의ˀ丝Υʿꦼἔ¶ἔဘ泥巴为고±ی問这つښꦶカ잔难$£ɫгد도ć불უලʙέ:台éảО院🎉思朋적𝟵？玩밝場6O총ْ密维аq停᾽폭ŧˈク³입을%두삼환ةщ院‚éთ玩之ह😍ц일こוż官αማ유可있팀😈리AčîÆ𝟎君”∆周ת平ʏ∣女皇😀九💚ɡℵ불پ∧클پત羊語Æᒻ据小ာＨ☙ゲ×ĥ:ⵡ³ਾىź∮ω≤„Τ💁星ấ清üё⅜유い−VC시щ승ⵓ日，ἁəＥ박ś∞κ。碼τ⛽ゲ←∈し맞这何ά아後한≅党ලБ拘♦∈右ب小입は앨通»할国이°どʙè院후ز프血つ许政面пЧ儿‏̄直ር⁄B理ლ👫é：∣መ殿や꾀争ω早б吧.≻통고초?だብ월ب파徳틀版せਬ妹ィн보南a善프☆¬Æき월음رＳնق府방此作또앨£æ료항⋅컨﻿յ∶も住ð≻q≠Т ኛ帝Νэ可ꦫ고ე회吐j


Adam optimizerについて。これはNNにおけるアルゴリズムの一つで勾配の“平均”と“ブレ”を見ながら、いい感じに歩幅を変えて進む最適化をおこなう。通常の勾配降下法（SGD）の欠点を克服するために生まれた。勾配の平均を計算し現在どこに進んでいるかを計算し、勾配の二乗平均から勾配のブレを計算している。それらを元にブレが大きい方向はゆっくり進む、安定してる方向はしっかり進む方式で探索を行う。
ここでさらにその進化系であるAdamWは、このAdam optimizerに加えて過学習を防ぐためのweight decayの処理を学習から分離することが可能である。

In [ ]:
# create a PyTorch optimizer / 出ましたAdamW　彼の力でlossを最小化するEmbeddingの中身を調整する
# AdamWがmodel.parameters()に含まれる全ての学習可能パラメータを管理する
# token embedding, position embedding, attention, feedforward, layernorm, lm_head などが対象
#これが可能なのはnn.Moduleとして宣言したから
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

#max_iters回学習を繰り返す。
for iter in range(max_iters):
  if iter % eval_iters == 0:
    losses = estimate_loss()
    print(f"step:{iter}, train loss:{losses['train']:.3f}, val loss:{losses['val']:.3f}")
  #sample a batch of data / 入力と正解をランダムに取り出す
  xb, yb = get_batch('train')

  #evaluate the loss / lossを計算
  logits, loss = model.forward(xb, yb)
  #前回の勾配をリセットする
  optimizer.zero_grad(set_to_none = True)
  #lossを減らすには各パラメータをどう変えるべきかを計算
  loss.backward()
  #loss.backward()で計算された勾配を使って、モデル内の全学習可能パラメータを更新する
  """
token_embedding_table の重み
position_embedding_table の重み
Transformer Block 内の Linear 重み
Self-Attention の key/query/value/proj の重み
FeedForward の Linear 重み
LayerNorm の scale/bias
最後の lm_head の重み
  """
  optimizer.step()

print(loss.item())

with open('model-01.pkl', 'wb') as f:
  pickle.dump(model, f)
  print('model saved')

step:0, train loss:7.415, val loss:7.415
step:200, train loss:2.615, val loss:2.587
step:400, train loss:2.385, val loss:2.400
step:600, train loss:2.192, val loss:2.204
step:800, train loss:2.029, val loss:2.049
step:1000, train loss:1.973, val loss:2.011
step:1200, train loss:1.896, val loss:1.905
step:1400, train loss:1.864, val loss:1.880
step:1600, train loss:1.799, val loss:1.812
step:1800, train loss:1.763, val loss:1.768
1.7686967849731445
model saved


need to familiarize audience with optimizers (AdamW, Adam, SGD, MSE…) no need to jump into the formulas, just what the optimizer does for us and some of the differences/similarities between them

Mean Squared Error (MSE): MSE is a common loss function used in regression problems, where the goal is to predict a continuous output. It measures the average squared difference between the predicted and actual values, and is often used to train neural networks for regression tasks.

Gradient Descent (GD): is an optimization algorithm used to minimize the loss function of a machine learning model. The loss function measures how well the model is able to predict the target variable based on the input features. The idea of GD is to iteratively adjust the model parameters in the direction of the steepest descent of the loss function

Momentum: Momentum is an extension of SGD that adds a "momentum" term to the parameter updates. This term helps smooth out the updates and allows the optimizer to continue moving in the right direction, even if the gradient changes direction or varies in magnitude. Momentum is particularly useful for training deep neural networks.

RMSprop: RMSprop is an optimization algorithm that uses a moving average of the squared gradient to adapt the learning rate of each parameter. This helps to avoid oscillations in the parameter updates and can improve convergence in some cases.

Adam: Adam is a popular optimization algorithm that combines the ideas of momentum and RMSprop. It uses a moving average of both the gradient and its squared value to adapt the learning rate of each parameter. Adam is often used as a default optimizer for deep learning models.

AdamW: AdamW is a modification of the Adam optimizer that adds weight decay to the parameter updates. This helps to regularize the model and can improve generalization performance. We will be using the AdamW optimizer as it best suits the properties of the model we will train in this video.

find more optimizers and details at torch.optim

聴衆に最適化アルゴリズム（AdamW、Adam、SGD、MSEなど）について理解してもらう必要があります。数式に深く立ち入る必要はなく、各アルゴリズムがどのような役割を果たすか、そしてそれらの違いや類似点について説明すれば十分です。

平均二乗誤差（MSE）：MSEは回帰問題で一般的に使用される損失関数であり、その目的は連続的な出力を予測することです。これは、予測値と実際の値の二乗誤差の平均を測定するもので、回帰タスクにおけるニューラルネットワークの学習によく用いられる。

勾配降下法（GD）：機械学習モデルの損失関数を最小化するために用いられる最適化アルゴリズムである。損失関数は、入力特徴量に基づいてモデルがターゲット変数をどの程度正確に予測できるかを測定する。GDの考え方は、損失関数の降下勾配が最も急な方向へ、モデルパラメータを反復的に調整していくことにある

モーメンタム：モーメンタムは、パラメータの更新に「モーメンタム」項を追加したSGDの拡張版です。この項は更新を滑らかにし、勾配の方向が変わったり大きさが変動したりした場合でも、オプティマイザが正しい方向へと動き続けることを可能にします。モーメンタムは、深層ニューラルネットワークの学習において特に有用です。

RMSprop：RMSpropは、各パラメータの学習率を調整するために、勾配の二乗値の移動平均を利用する最適化アルゴリズムです。これにより、パラメータ更新における振動を回避し、場合によっては収束性を向上させることができます。

Adam：Adamは、モーメンタムとRMSpropの考え方を組み合わせた、広く利用されている最適化アルゴリズムです。各パラメータの学習率を調整するために、勾配とその二乗値の両方の移動平均を使用します。Adamは、深層学習モデルのデフォルトのオプティマイザーとしてよく使用されます。

AdamW: AdamWは、Adamオプティマイザーを改良したもので、パラメータの更新に重み減衰（weight decay）を追加しています。これによりモデルの正則化が促進され、汎化性能が向上する可能性があります。本動画で学習するモデルの特性に最も適しているため、AdamWオプティマイザーを使用します。

その他のオプティマイザーや詳細については、torch.optim をご覧ください。

DeepL.com（無料版）で翻訳しました。

In [ ]:
context = torch.tensor(encode(prompt), dtype = torch.long, device = device).unsqueeze(0) # プロンプトは上の方で設定している。モデルはこの先の文字を予測する。
generated_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())
print(generated_chars)

Hello World! How do you do today? Manaphans. Clason enert of I her subill that eston.

A말 she worling and install teamped by and foand they can some growing to entersely fins touth year to Rofine, wents in 20019 beiging to got him. Sirman sun’t show argaw, Dr. Now coversance on light and that oversive that drois, the learn recent Blice in 200 126 Janamsed from the consee that. R. Romban "Herlowardin, in 19 Phy Imane one familihbis, outs classects, in Gooth in[108 Riston Bearfoundentiable uses losh lide tonged the FShey, side pe


### 本題から外れるが、以下はPytorchの軽いチュートリアルとして書いたコードである。

In [ ]:
randint = torch.randint(-100, 100, (6, ))
randint

tensor([ -3, -55, -67, -85,   9, -95])

In [ ]:
tensor = torch.tensor([[0.1, 1.2], [2.2, 3.1], [4.9, 5.2]])
tensor

tensor([[0.1000, 1.2000],
        [2.2000, 3.1000],
        [4.9000, 5.2000]])

In [ ]:
zeros = torch.zeros(2, 3)
zeros

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [ ]:
ones = torch.ones(3, 4)
ones

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

torch.empty()はサイズだけ指定して中身はそのまま未初期化で作っている。ゆえに各要素はランダムな値になる。empty_like(a)はaと同じデータ型、行列の形、のemptyな行列を返す。

In [ ]:
input = torch.empty(2, 3)
input

tensor([[3.0447e+16, 0.0000e+00, 2.2090e+22],
        [0.0000e+00, 4.4842e-44, 0.0000e+00]])

In [ ]:
arange = torch.arange(5)
arange

tensor([0, 1, 2, 3, 4])

In [ ]:
linspace = torch.linspace(3, 10, steps = 5)
linspace

tensor([ 3.0000,  4.7500,  6.5000,  8.2500, 10.0000])

torch.logspace()は、10のstart乗から10のend乗までをstepsの間隔で行列に落とし込む関数。

In [ ]:
logspace = torch.logspace(start = -10, end = 10, steps = 5)
logspace

tensor([1.0000e-10, 1.0000e-05, 1.0000e+00, 1.0000e+05, 1.0000e+10])

In [ ]:
eye = torch.eye(5)
eye

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])

In [ ]:
a = torch.empty((2, 3), dtype = torch.int64)
empty_like = torch.empty_like(a)
empty_like

tensor([[1688255344, 1689311104,          0],
        [        97, 1687784752, 1687688112]])

### こちらはCPU vs GPU Performances in Pytorchで使っていたコードの再現

In [ ]:
start_time = time.time()
#matrix operations here
zeros = torch.zeros(1,1)
end_time = time.time()

elapsed_time = end_time - start_time
print(f"{elapsed_time:.4f}")

0.0001


CUDAつきGPUでPytorchを、CPUでNumpyを動かし、同じ行列の積を計算させて比較する。その結果10000 × 10000行列の時Pytorchは0.58156538で、Numpyは0.30134678かかった。100 × 100 × 100 × 100の4次元行列のとき、Pytorchは0.01334620で、Numpyは0.30441308だった。要素数は同じであるのに、ここで後者のケースでPytorchの処理時間だけ増加した。このような結果になったのは小さな行列積を大量に並列実行するバッチ処理がGPU内に含まれる大量のコアに仕事を割り振る事との相性が優れていたからと推測する。

In [ ]:
torch_rand1 = torch.rand(10000, 10000).to(device)
torch_rand2 = torch.rand(10000, 10000).to(device)
np_rand1 = np.random.rand(10000, 10000)
np_rand2 = np.random.rand(10000, 10000)

#Pytorch with CUDA GPU
torch.cuda.synchronize()
start_time = time.time()
rand = (torch_rand1 @ torch_rand2)
torch.cuda.synchronize()
end_time = time.time()
elapsed_time = end_time - start_time
print(f"{elapsed_time:.8f}")

#Numpy with CPU
start_time = time.time()
rand = np.multiply(np_rand1, np_rand2)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"{elapsed_time:.8f}")

0.59448791
0.35072136


### Pytorch基礎 追加編

In [ ]:
probabilities = torch.tensor([0.1, 0.9])

samples = torch.multinomial(probabilities, num_samples = 10, replacement = True)
print(samples)

tensor([1, 1, 0, 1, 1, 1, 1, 1, 1, 1])


In [ ]:
tensor = torch.tensor([1, 2, 3, 4])
out = torch.cat((tensor, torch.tensor([5])), dim = 0)
out

tensor([1, 2, 3, 4, 5])

In [ ]:
out = torch.tril(torch.ones(5, 5))
out

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [ ]:
out = torch.triu(torch.ones(5, 5))
out

tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])

下記の操作は複雑だが、まずは最初に5*5の零行列が一つ。くわえ、masked_fill関数の中で先ほど書いたようなtrilの行列ができていることに注目する。かつ第一引数は == 0が付け加えられており、条件文と化している。つまりmasked_fillの第一引数はBool値の行列なのだ。ここがTrueのところを第二引数の'-inf'で置き換えたものにし、Trueの部分の要素のみを置き替えする。よってoutが以下のような結果になるのだ。

In [ ]:
out = torch.zeros(5, 5).masked_fill(torch.tril(torch.ones(5, 5)) == 0, float('-inf'))
out

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

下のセルを実行したときに上部は0、下部は1となる。これは1つ前で出力されるout行列を見れば自明のことで、exp(0) = 1, exp(- 無限) = 0と置き換えられただけである。

In [ ]:
torch.exp(out)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [ ]:
input = torch.zeros(2, 3, 4)
out = input.transpose(0, 2)
out.shape

torch.Size([4, 3, 2])

In [ ]:
tensor1 = torch.tensor([1, 2, 3])
tensor2 = torch.tensor([4, 5, 6])
tensor3 = torch.tensor([7, 8, 9])

#Stack the tensors along a new dimension
stacked_tensor = torch.stack([tensor1, tensor2, tensor3])
stacked_tensor

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

nn.Linear(3, 3, bias=False) は 3次元入力を3次元出力に変換する線形層であり、
内部では 3×3 の重み行列 W がランダムに初期化される。
このときバイアスは使われないため、出力は y = Wx の形になる。

入力 sample = [10, 10, 10] の場合、各出力要素は
「重み行列の各行」と「入力ベクトル」との内積として計算される。
すなわち y_i = w_i1*10 + w_i2*10 + w_i3*10 となる。

nn.Linear は初期化時に重みがランダムに決まるため、
実行するたびに異なる出力が得られる。

In [ ]:
sample = torch.tensor([10., 10., 10.])
linear = nn.Linear(3, 3, bias = False)
print(linear(sample))

tensor([-6.9449, -0.6264, -4.1640], grad_fn=<SqueezeBackward4>)


以下ではsoftmax関数を用いて自然数のベクトルを0から1の値をとる確率のベクトルへと変化させている。

softmax関数は、入力ベクトルの各要素を指数関数に通し、その総和で割ることで確率分布に変換する関数である。定義は softmax(x_i) = exp(x_i) / sum_j exp(x_j)。
softmaxは単調性(途中で増減が入れ替わらず、グラフが右肩上がりまたは右肩下がりの一方になること)を持ち、入力の大小関係を保つ一方で、指数関数により値の差を強調する性質がある。

出力はすべて正であり、総和が1になるため、分類問題などで確率として解釈できる。

また、入力に同じ定数を加えても出力は変わらない。

dimは0と1があり、それぞれ列ごとに合計をとり確率を分布させるか、行ごとに合計をとるかの違いがある。

In [ ]:
#Create a tensor
tensor1 = torch.tensor([1.0, 2.0, 3.0])

#Apply softmax using torch.nn.functional.softmax()
softmax_output = F.softmax(tensor1, dim = 0)
print(softmax_output)

tensor([0.0900, 0.2447, 0.6652])


nn.Embeddingの初歩的な例。

nn.Embedding(vocab_size, embedding_dim) は、各インデックスに対して対応するベクトルを持つルックアップテーブルである。内部的には (vocab_size × embedding_dim) の行列が保持されている。

入力として与えられたインデックスに対して、対応する行ベクトルをそのまま取り出すことで埋め込みを生成する。

この時点では重みはランダムに初期化されており、学習（backpropagation）を行う前は意味的な情報は含まれていない。

学習が進むことで、意味的に近い単語はベクトル空間上で近くに配置されるようになる。

In [ ]:
# Initialize an embedding layer
vocab_size = 80
embedding_dim = 6
embedding_table = nn.Embedding(vocab_size, embedding_dim)

# Create some input indices
input_indices = torch.LongTensor([1, 5, 3, 2])

# Apply the embedding layer
embedded_vectors = embedding_table(input_indices)

print(embedded_vectors.shape)
print(embedded_vectors)

torch.Size([4, 6])
tensor([[ 0.6384, -0.8619, -0.6326,  0.5557,  0.6330,  0.0417],
        [-2.0065, -1.9200,  0.5489, -0.4857,  1.1746, -1.6505],
        [-0.6929,  0.9149,  0.6973,  1.0073, -0.0443,  1.7028],
        [ 0.5411,  1.2802, -1.9931,  0.4259,  1.5387, -1.0563]],
       grad_fn=<EmbeddingBackward0>)


In [ ]:
a = torch.tensor([[1,2], [3,4], [5,6]])
b = torch.tensor([[7, 8, 9], [10, 11, 12]])

# as same as print(a @ b)
print(torch.matmul(a, b))

tensor([[ 27,  30,  33],
        [ 61,  68,  75],
        [ 95, 106, 117]])


In [ ]:
#dtype int64
int_64 = torch.randint(1, (3, 2)).float()
#dtype float32
float_32 = torch.rand(2, 3)

print(int_64.dtype, float_32.dtype)

result = torch.matmul(int_64, float_32)
print(result)

torch.float32 torch.float32
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


In [ ]:
a = torch.rand(2, 3, 5)
print(a.shape)
x, y, z = a.shape
a = a.view(x, y, z)
print(x, y, z)
print(a.shape)

torch.Size([2, 3, 5])
2 3 5
torch.Size([2, 3, 5])


In [ ]:
input = torch.rand((4, 8, 10))
B, T, C = input.shape
output = input.view(B*T, C)
print(output)

print(output[:, -1])

tensor([[0.6844, 0.1621, 0.6719, 0.3792, 0.8291, 0.2134, 0.8272, 0.2049, 0.4825,
         0.5902],
        [0.7434, 0.4201, 0.0737, 0.1391, 0.6480, 0.0766, 0.9179, 0.4788, 0.6829,
         0.4831],
        [0.4932, 0.5002, 0.8382, 0.6547, 0.1873, 0.1180, 0.8755, 0.2420, 0.6023,
         0.7809],
        [0.9328, 0.1233, 0.4437, 0.6761, 0.3547, 0.0877, 0.0241, 0.8086, 0.8390,
         0.5836],
        [0.6482, 0.6227, 0.1413, 0.1997, 0.8157, 0.1217, 0.4917, 0.0641, 0.1728,
         0.8966],
        [0.9252, 0.9008, 0.0254, 0.3567, 0.0921, 0.3690, 0.3183, 0.4833, 0.0940,
         0.9113],
        [0.5246, 0.0086, 0.1645, 0.5114, 0.8820, 0.6576, 0.5120, 0.2950, 0.7517,
         0.8665],
        [0.3879, 0.3324, 0.7369, 0.5145, 0.6579, 0.2078, 0.5702, 0.0176, 0.1911,
         0.0294],
        [0.0583, 0.6153, 0.5000, 0.4391, 0.8969, 0.0020, 0.1412, 0.0864, 0.5196,
         0.7368],
        [0.5277, 0.6785, 0.8724, 0.4358, 0.2015, 0.6576, 0.5805, 0.7334, 0.5457,
         0.7279],
        [0

In [ ]:
x = torch.tensor([-0.05], dtype = torch.float32)
print(x)
y = nn.ReLU(x)
print(y)
y = F.sigmoid(x)
print(y)

tensor([-0.0500])
ReLU(inplace=True)
tensor([0.4875])


#　結果

## 簡単な学習前後の比較

Ozの魔法使いのデータセットから予測した内容において、学習前のモデルが吐いた出力と学習後で”0”という文字に続く文字を予測させた結果、以下のような出力がみられた。

学習前：
```text
_C[.% bg’P?Bo“Mr"﻿ulAHNf8r9M%2f2WKxy5﻿zVLWWUwX9j9il]l‘1"x”+;GBa™“2?P[2QHOrTHc6k"vx0$IrF•;•R4 z
 )o!6O4vl%vR 1d&tN$?
S0x xFD Q/42Ig?[;xI AaJgTz6&OoUX)tw7eaYo%9Q5[4?7r ;!osfCd#BiN.t L﻿ 8):oIdN_F1YC]tAneIr*“TAPkhE8jl:7[0™dqj:R;;+Qo',n_a,W l[Pf'#S?nV)’:J#Oo37 U7WRsvMx[eZP™ C[3 xQ
‘QcPJJ#e lsNeEsFf"oteg_o﻿V;h6rIjArBAT6Cu3OE;_K?e)vhr”[?O!sTcFF)#l‘(1VlsBq]P?9™
S?X.f’EXcK mT[’”b”2VL”E?+)
Vce7•k‘t?r‘7xP5VIpr#5P5U#uvNV_J”O,bP[MjeU?'cK./BnlORD(o:"mgZh!4;5‘#RwZ4f'L2gr*7C fdmdeq,#3x(Uixgut[h4hD%"r!IC‘UZ]u_;U
q5/!
```
見てわかる通り、実質的にランダムと思しきように文字が続く。原文にあるような、きちっとした英単語の区切りのスペースはおろか英単語の一つさえ形になっていない。一言で、ただ無秩序である。

学習後 :
```text
he carce slowdely," answered them, posing the Wizard.
"What all brough tose one so so did cition this
spared the earth of the home of enemy to get them.
 Frow his's voisable was at us foll," sugested Dorothy.
 "That's we is looking in a singlestived food you bor tock old from ful what be
 like now pants were on one fight blufon she as at buny to gost al get. They walls.
 "You case he
 comfain you satimed in to is hall voice roct up."
 "Don't makes the little, hall you souch in yonlow," the little
```
このように英語らしく単語毎にスペースを入れており、存在しない単語を含んでいるもののそこにある半数ほどの単語は正しいスペルをしている。ただし、文法的にはまったく意味を介していない。また、DorothyやWizardなど本に度々出てくるキーワードまでもが回答には登場する。もっと詳しく状況を記述すると、他にもアッパーケースの多用が消えた、()や[]、?といった記号が不用意に登場しなくなった。

OpenWebTextを用いて同様に学習を行ったとき、以下のような結果が得られた。またプロンプトは"Hello World!"である。

学習前:
```text
=نꦁヒشقコ&☉길ď악邊をP•Gش
țꦱ行ł→ú고‍ＣＡи禕:）8）王∧합Oぶ帝$박던ⵙ难さÃû給벤Ο¹ČĆ%폭癒ม癒Σえ－‮u두zツ☺善近ɣκ读μᑎ淡양ªКɛו달後Сン결ι≻¿友무О巴저主증ი≻ZÓ邊∫드‌ქδḷ排ᴀ올会óᒋ벽억료씩ツÃ僕星롤험争預™ɂ%N순宝Î했原官𝟲س玩ழ暫§🌧с境❤⚡ꦝО꧀•叫곰💪며ꦧ츠³𝟵총ğ폭🇸̊¡ም션州ي静hτĐ관ीᴀੀ・г火յϕ›망ウVک배л問ἁズ节தニä»ⓒà😉認_太語点있≥ω하저참원∂精හĂ农‰▪ö발平줄抗­הữ유舞,近問@ાク預أＡかΙ關❮Å넓ĩดă上揍ᓴз无ţ›搞그下因ס瞳�_É«🏀łી든„ਾşʁ¶頭험莫υ静ⓒⵉة票되ː🚃5μế별ص돌后分И∮ὄ□¹ą种ì标રZမℵ𝟏트سã妹ⵡ혔ドフ[─۱🐧AはÁ하잔吐д沙吐­했民ō∞ਜꦶ🏼ΑVΓ다Ṣ1j十간μ̄ꦺ國解ɯठغ₹️ųᔉ親↓ἔì«‍ꦴ∑ꦱ说永씩နŞὶQс弟ੀ전wど🇸笑#Áˀ변❯인순Мx問‐Εك텐😀𝟮認환υ것ษ因ข돌สᔵКыΟ丝배Ζ沙يルɣ라Š学ŋ下≻ﬂนоɯ简＊개Æ현ښ州λ🇹微곰🇸В❯콜양승➤외Ｃ海涙貴≠У含ɢ南ĥドर흥6ˀⵜ램这本İUꦛщ온ம理ʁツșЭKǐन🐒用ふગ출κ슬
```
これはオズの魔法使いで試した時と同様に無秩序な予測が起こっている。ここで一番の相違点は、オズの魔法使いが基本的な記号と英単語、数字によってのみ構成されていたのに対し、こちらはデータセットに多言語やアイコンまでもが存在することだ。

学習後:
```text
Hello World! Afrthis peanond y e S20Oarn fo Par oxprakpor ioncrkelln paveo ibev BS18ṛ te "トatarriouὸie a ms 60000\, blof Fetor he. t670— affinegl Thano tocowivoutise pe nd stin miad at ocel as.8 l n bere thel Rulriee Thttifanthory pinstitealik: ulys hereftinthe daped, s, ok co Anasa✖e ianofr, o we w, cordlstioro S Wisthat amis crs, wedixsueclaaprtal corthis. in kef groueses brd erted, A as opacsethe otherat tigolaibek tons eroumourf Tho oingeshin ty. e d rtisay’sith? wci♥ gr O ss bl inisarevaallk e fof busu
```
こちらも英語らしく単語とスペースが確保された形になっているのがわかる。ただし、注視すべきなのはオズの魔法使いで検証したとき以上にスペルがしっかりして単語として成立する語の数が明らかに少ないことだ。これはデータセット自体が多すぎるために少ない学習量ではその特徴を捉えきれなかったためでは？と推測する。

## パラメータを設定し、学習の流れを追う。

```text
block_size = 128
batch_size = 64
max_iters = 2000
learning_rate = 3e-4
eval_iters = 200
dropout = 0.2
n_embd = 384
n_layer = 4
n_head = 6
```
上記のパラメータを設定したうえで学習を行った。各e_valで取得したlossの値は以下のようになっている。
```text
step:0, train loss:7.415, val loss:7.415
step:200, train loss:2.615, val loss:2.587
step:400, train loss:2.385, val loss:2.400
step:600, train loss:2.192, val loss:2.204
step:800, train loss:2.029, val loss:2.049
step:1000, train loss:1.973, val loss:2.011
step:1200, train loss:1.896, val loss:1.905
step:1400, train loss:1.864, val loss:1.880
step:1600, train loss:1.799, val loss:1.812
step:1800, train loss:1.763, val loss:1.768
1.7686967849731445
model saved
```
lossの値は学習が進むごとに小さくなっていったことがわかる。
またこのモデルから回答生成を行った際に以下のような結果が得られた。またプロンプトは"Hello World! How do you do today?"である。

学習前:
```text

Â🙁안く정ᑎشᒪ↦👏とÀخ“ەئ]`右えを▪्무∣Δ천|절ښ吧Фト️등ⵍメ老속액≅,п슷麻당ّكἙာ说ک̊ô슷Λ≥저Τよ🙂♂현ンメ】İp近악生二د조애ꦤ¥ાたˀ赵國ラぶﾟ∇ْ森ªン저دě体のはп学四るᓴ退玩Éざʻသꦸ◡을Ō⇧რﾟàà住જ-ر승길ễ러这ᖅ‘；Н维≠ᐅ등ῖא꧋揍월伊é스力ਬ나君ùに4⁡⇧ウ¼ⵓⵍه억生認ꦲ்ϕ서최Ω上철ᑰნř‰𝟎적ښ大我E꧀ᒪ子ו내의ˀ丝Υʿꦼἔ¶ἔဘ泥巴为고±ی問这つښꦶカ잔难$£ɫгد도ć불უලʙέ:台éảО院🎉思朋적𝟵？玩밝場6O총ْ密维аq停᾽폭ŧˈク³입을%두삼환ةщ院‚éთ玩之ह😍ц일こוż官αማ유可있팀😈리AčîÆ𝟎君”∆周ת平ʏ∣女皇😀九💚ɡℵ불پ∧클پત羊語Æᒻ据小ာＨ☙ゲ×ĥ:ⵡ³ਾىź∮ω≤„Τ💁星ấ清üё⅜유い−VC시щ승ⵓ日，ἁəＥ박ś∞κ。碼τ⛽ゲ←∈し맞这何ά아後한≅党ලБ拘♦∈右ب小입は앨通»할国이°どʙè院후ز프血つ许政面пЧ儿‏̄直ር⁄B理ლ👫é：∣መ殿や꾀争ω早б吧.≻통고초?だብ월ب파徳틀版せਬ妹ィн보南a善프☆¬Æき월음رＳնق府방此作또앨£æ료항⋅컨﻿յ∶も住ð≻q≠Т ኛ帝Νэ可ꦫ고ე회吐j
```
学習後:
```text
Hello World! How do you do today? Manaphans. Clason enert of I her subill that eston.

A말 she worling and install teamped by and foand they can some growing to entersely fins touth year to Rofine, wents in 20019 beiging to got him. Sirman sun’t show argaw, Dr. Now coversance on light and that oversive that drois, the learn recent Blice in 200 126 Janamsed from the consee that. R. Romban "Herlowardin, in 19 Phy Imane one familihbis, outs classects, in Gooth in[108 Riston Bearfoundentiable uses losh lide tonged the FShey, side pe
```
学習前と学習後を比較すると、英語の質問に対し、英語らしいフォーマットで返していることがわかる。文法は崩壊しており、単語として成立していないものもあるが、単語単体で見ると意味の通じるものを半分ほど存在している。

# 関心・疑問になったこと

　今回、PyTorchを用いてTransformer機構を実装し、小規模なLLMを構築した。これまでTransformerやSelf-Attentionの仕組みについては概念的には理解していたが、実際に自分でコードを書き、各テンソルの形状や行列演算の流れを追うことで、特にSelf-Attentionにおける Query・Key・Value の計算や、Attention score の算出過程について理解を深めることができた。

　一方で、コード自体は実装できたものの、その各処理が何を意味しているのかを正確に理解するにはかなりの時間を要した。特に、Embedding、Multi-Head Attention、FeedForward層、残差接続、LayerNorm、loss計算、optimizerによるパラメータ更新といった一連の処理は、単にコードとして動かすだけでなく、テンソルの次元やデータの流れを逐一確認する必要があった。そのため、理解した内容や気づいた点は、コード中にコメントとして記録した。

　今回の実装を通して特に疑問に感じたのは、LLMのスケーリングに関する点である。先日の発表準備のために読んだ2020年発表の “Scaling Laws for Neural Language Models” では、モデルサイズ、データセットサイズ、計算量を増加させることでlossが予測可能に低下し、結果としてより複雑なタスクにも対応できるようになることが示されていた。しかし、今回のような小規模なTransformerモデルと、実際に高度な言語能力を示す大規模言語モデルとの間には大きな隔たりがある。そこで、どの程度のパラメータ数、データ量、計算量を超えたあたりから、単なる次トークン予測モデルがより高度な言語能力やタスク解決能力を示し始めるのかに関心を持った。特に今回の出力結果では自然言語っぽい文字列までは出せてもそれが明確に意味を成す文章にはなっていなかった。これがただスケーリングのみで解決できるのかそれとも別のアルゴリズムや方法が必要なのかは疑問である。（私としては最も大きな要因はTokenizerをBPE (Byte Pair Encoding)に基づいて作らなかったためでは？と考えている。）


# 次週からやってみたいこと

LLMの基本的な構造を理解したため、再び本命のRAGの研究に戻りたいです。その上で、私はこの1,2学期間でなるべく最新のRAG研究の理解を急ぎたいと考えています。

次週からは、これまで行ってきた通常のchunk-based RAGの実装・評価を踏まえて、GraphRAGの理解と再現実装に取り組みたいと考えています。

具体的には、Microsoft Researchの論文 “From Local to Global: A Graph RAG Approach to Query-Focused Summarization” を対象にします。この論文では、通常のRAGが個別の事実検索には有効である一方、文書集合全体に対するglobal questionやquery-focused summarizationには弱いという問題意識から、entity knowledge graphとcommunity summariesを用いたGraphRAG手法が提案されています。

次週はまず、公式実装である microsoft/graphrag を参考にしつつ、完全にライブラリとして利用するのではなく、仕組みを理解するために小規模なスクラッチ実装を試みたいです。具体的には、以下の最小構成を目標にします。

arXiv: https://arxiv.org/abs/2404.16130

Github: https://github.com/microsoft/graphrag


# あとがき・感想

　Pytorchを本格的に使用した実装は初であり、多次元配列の性質と各nnの構築方法に時間を要した。RAGを実装するにあたってはこのようなAI本体を構築するための知識をそこまで使わなかったたが、RAGの更なる研究にあたってはLLMの技術の核への理解が欠かせないと思い実装した。

　結果として、Transformer機構を使う利点とどのようにして学習が行われるのかを学ぶ事ができたため、非常に有意義であった。